In [8]:
!pip install wandb gensim -q
import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')

import wandb
wandb.login(key="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")
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

OPTS = ['A', 'B', 'C', 'D', 'E']

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sub   = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train:", train.shape)
print("test :", test.shape)
train.head(3)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


train: (2000, 8)
test : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [9]:
print("=== null counts ===")
print(train.isnull().sum())

print("\n=== answer distribution ===")
print(train['answer'].value_counts())

print("\n=== prompt length (words) ===")
print(train['prompt'].str.split().str.len().describe().round(2))

print("\n=== avg option length (words) ===")
for o in OPTS:
    print(f"  {o}: {train[o].str.split().str.len().mean():.1f}")

=== null counts ===
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

=== answer distribution ===
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

=== prompt length (words) ===
count    2000.00
mean       18.15
std         6.78
min         3.00
25%        14.00
50%        17.00
75%        22.00
max        51.00
Name: prompt, dtype: float64

=== avg option length (words) ===
  A: 26.1
  B: 26.5
  C: 26.5
  D: 26.0
  E: 26.2


In [10]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

def remove_stopwords(tokens):
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 1]

for col in ['prompt'] + OPTS:
    train[f'clean_{col}'] = train[col].apply(clean_text)
    test[f'clean_{col}']  = test[col].apply(clean_text)

# vocab size of cleaned prompts
all_words = []
for p in train['clean_prompt']:
    all_words.extend(p.split())

print(f"unique vocab in cleaned prompts: {len(set(all_words))}")
print(f"sample — original : {train['prompt'].iloc[0]}")
print(f"sample — cleaned  : {train['clean_prompt'].iloc[0]}")

unique vocab in cleaned prompts: 859
sample — original : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
sample — cleaned  : pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options


In [11]:
# build corpus from train + test so vocab covers inference time too
full_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        full_corpus.append(row['prompt'])
        for o in OPTS:
            full_corpus.append(row[o])

tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=25000, sublinear_tf=True)
tfidf.fit(full_corpus)
print(f"tfidf vocab size: {len(tfidf.vocabulary_)}")

# word2vec on cleaned + stopword-removed tokens
w2v_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        w2v_corpus.append(remove_stopwords(tokenize(row['prompt'])))
        for o in OPTS:
            w2v_corpus.append(remove_stopwords(tokenize(row[o])))

w2v_model = Word2Vec(
    sentences=w2v_corpus, vector_size=150, window=7,
    min_count=2, workers=4, sg=1, epochs=15, seed=42
)
print(f"w2v vocab size : {len(w2v_model.wv)}")
print(f"vector dims    : {w2v_model.vector_size}")

tfidf vocab size: 10915
w2v vocab size : 2883
vector dims    : 150


In [12]:
def tfidf_cosine(text1, text2):
    v1 = tfidf.transform([text1])
    v2 = tfidf.transform([text2])
    return float(cosine_similarity(v1, v2)[0][0])

def avg_w2v(text):
    tokens = remove_stopwords(tokenize(text))
    vecs   = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)

def w2v_cosine(text1, text2):
    v1 = avg_w2v(text1).reshape(1, -1)
    v2 = avg_w2v(text2).reshape(1, -1)
    if np.all(v1 == 0) or np.all(v2 == 0):
        return 0.0
    return float(cosine_similarity(v1, v2)[0][0])

def rank_tfidf(row):
    sims = {o: tfidf_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_w2v(row):
    sims = {o: w2v_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p == actual:
            hits  += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, predictions, k=3):
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

actuals     = train['answer'].tolist()
top3_maj    = list(train['answer'].value_counts().index[:3])
map3_maj    = round(mapk(actuals, [top3_maj] * len(train)), 4)
tfidf_preds = [rank_tfidf(row)[:3] for _, row in train.iterrows()]
map3_tfidf  = round(mapk(actuals, tfidf_preds), 4)
w2v_preds   = [rank_w2v(row)[:3] for _, row in train.iterrows()]
map3_w2v    = round(mapk(actuals, w2v_preds), 4)

print(f"majority baseline MAP@3 : {map3_maj}")
print(f"tfidf cosine     MAP@3  : {map3_tfidf}")
print(f"word2vec cosine  MAP@3  : {map3_w2v}")

run = wandb.init(
    project = "23f1000362-t22026",
    name    = "milestone1-tfidf-w2v-baselines",
    config  = {
        "milestone": 1, "tfidf_ngram": "(1,2)",
        "tfidf_max_feats": 25000, "w2v_vector_size": 150,
        "w2v_window": 7, "w2v_sg": 1, "w2v_epochs": 15
    },
    tags    = ["milestone-1", "baseline"]
)
wandb.log({
    "map3/majority_baseline" : map3_maj,
    "map3/tfidf_cosine"      : map3_tfidf,
    "map3/word2vec_cosine"   : map3_w2v,
    "model/tfidf_vocab"      : len(tfidf.vocabulary_),
    "model/w2v_vocab"        : len(w2v_model.wv)
})
wandb.finish()

# submission — always named submission.csv
test_preds = [" ".join(rank_tfidf(row)[:3]) for _, row in test.iterrows()]
submission = pd.DataFrame({
    'ID'        : test['id'].values,
    'Prediction': test_preds
})
submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv saved")
submission.head(5)

majority baseline MAP@3 : 0.4213
tfidf cosine     MAP@3  : 0.3228
word2vec cosine  MAP@3  : 0.3024


map3/majority_baseline,▁
map3/tfidf_cosine,▁
map3/word2vec_cosine,▁
model/tfidf_vocab,▁
model/w2v_vocab,▁
map3/majority_baseline,0.4213
map3/tfidf_cosine,0.3228
map3/word2vec_cosine,0.3024
model/tfidf_vocab,10915
model/w2v_vocab,2883



submission.csv saved


,ID,Prediction
0,1,A B C
1,2,A B C
2,3,D A C
3,4,A E C
4,5,C A D
